## DNN Serving System
`togsim_configs/togsim_config_serving.yml` prints TOGSim's periodic stats every 10^6 cycles
instead of 10^4, so a model-long simulation keeps a readable log.

In [ ]:
import os
os.environ["TORCHSIM_TARGET"] = os.path.join(os.getcwd(), "togsim_configs", "togsim_config_serving.yml")
os.environ["TORCHSIM_LOG_PATH"] = os.path.join(os.getcwd(), "togsim_results")
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "outputs")
import glob
import torch

def latest_log():
    """The newest TOGSim log under TORCHSIM_LOG_PATH."""
    return max(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")), key=os.path.getmtime)

_seen = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")))

def new_logs():
    """The TOGSim logs written since the last call, oldest first: one per simulation, two for a
    resident mm (its operand's copy-in, then its body), whose cycles add up."""
    logs = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log"))) - _seen
    _seen.update(logs)
    return sorted(logs, key=os.path.getmtime)

### Scheduler
`torch.npu.TOGSimulator()` opens one TOGSim for the block; `torch.npu.launch_model` queues a model on a
stream at a timestamp in nanoseconds. ResNet18 has kernels that read runtime attributes, which only a
functional run records, so this machine file runs both modes.

In [ ]:
from torchvision.models import resnet18

device = torch.device("npu:0")

model = resnet18().eval()
input = torch.randn(1, 3, 224, 224, dtype=torch.bfloat16).to(device=device)
opt_fn = torch.compile(dynamic=False)(model.to(device, dtype=torch.bfloat16, memory_format=torch.channels_last))

# Run scheduler
with torch.npu.TOGSimulator(), torch.no_grad():
    torch.npu.launch_model(opt_fn, input, stream_index=0, timestamp=0)
    torch.npu.synchronize()

print("ResNet18 Simulation Done")

In [ ]:
for log in new_logs():
    !grep "Total execution cycles" {log}

### Load Generator
Requests arrive as a Poisson process. One ResNet18 takes about 21 ms on this machine, so at 100
requests/s over 20 ms a request that arrives while another runs waits in the stream's queue.

In [ ]:
import numpy as np


def poisson_request_generator(lambda_requests, max_msec_time=None):
    """Yield synthetic arrival times in milliseconds (first sample is 0)."""
    current_time = 0.0
    yield 0
    while max_msec_time is None or current_time < max_msec_time:
        current_time += np.random.exponential(scale=1000 / lambda_requests)
        if max_msec_time is not None and current_time > max_msec_time:
            break
        yield current_time


np.random.seed(0)
model0_lambda = 100.0  # requests per second
max_time_msec = 20.0

events = []
x = torch.randn(1, 3, 224, 224, dtype=torch.bfloat16, device=device)
for t in poisson_request_generator(model0_lambda, max_msec_time=max_time_msec):
    events.append((t, 0, opt_fn, (x,)))  # stream_index 0 → queue / partition 0
print(f"{len(events)} requests at", [round(t, 2) for t, *_ in events], "msec")

with torch.npu.TOGSimulator(), torch.no_grad():
    for t_msec, stream_index, model, args in events:
        torch.npu.launch_model(
            model,
            *args,
            stream_index=stream_index,
            timestamp=int(t_msec * 1e6),
        )  # ms → ns
    torch.npu.synchronize()

In [ ]:
for log in new_logs():
    !grep "Total execution cycles" {log}